# 06 — Visual Audit

**Purpose:** Generate annotated images for qualitative verification:
- **RESCUES:** Baseline missed, ensemble recovered
- **REGRESSIONS:** Baseline found, ensemble lost
- **BOTH_MISSED:** Neither detected
- **GHOSTS:** High-confidence ensemble predictions with no GT match

In [1]:
import os, json, shutil
import cv2
import numpy as np
from pycocotools.coco import COCO
from tqdm import tqdm

DATA_ROOT = r"C:\Users\dadab\Desktop\Clean Version\data"
OUTPUT_DIR = "outputs/"
IMG_ROOT = os.path.join(DATA_ROOT, "test/images")
GT_PATH = os.path.join(DATA_ROOT, "test/test_annotations.json")
BASELINE_JSON = os.path.join(OUTPUT_DIR, "test_results_exp01_baseline_best.json")
ENSEMBLE_JSON = os.path.join(OUTPUT_DIR, "test_results_ensemble_grand.json")

VIS_DIR = "figures/visual_audit/"
DIRS = {k: os.path.join(VIS_DIR, k) for k in ["RESCUES","GHOSTS","BOTH_MISSED","REGRESSIONS"]}

IOU_THRESH = 0.50
CONF_THRESH = 0.50
GHOST_THRESH = 0.85

In [2]:
def get_iou(a, b):
    xA, yA = max(a[0],b[0]), max(a[1],b[1])
    xB, yB = min(a[0]+a[2],b[0]+b[2]), min(a[1]+a[3],b[1]+b[3])
    inter = max(0,xB-xA)*max(0,yB-yA)
    union = a[2]*a[3]+b[2]*b[3]-inter
    return inter/union if union>0 else 0

def draw_label(img, text, x, y, fg=(255,255,255), bg=(0,0,0)):
    font, scale, thick = cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2
    (tw, th), _ = cv2.getTextSize(text, font, scale, thick)
    ty = y-10 if y >= 25 else y+th+15
    cv2.rectangle(img, (x, ty-th-5), (x+tw, ty+5), bg, -1)
    cv2.putText(img, text, (x, ty), font, scale, fg, thick)

## Generate Visual Audit Images

In [3]:
# Setup output dirs
if os.path.exists(VIS_DIR): shutil.rmtree(VIS_DIR)
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

coco = COCO(GT_PATH)
CLASS_MAP = {c['id']: c['name'] for c in coco.loadCats(coco.getCatIds())}

with open(BASELINE_JSON) as f: base_raw = json.load(f)
with open(ENSEMBLE_JSON) as f: ens_raw = json.load(f)

base_map, ens_map = {}, {}
for p in base_raw:
    if p['score'] >= CONF_THRESH: base_map.setdefault(p['image_id'], []).append(p)
for p in ens_raw:
    if p['score'] >= CONF_THRESH: ens_map.setdefault(p['image_id'], []).append(p)

counts = {"rescues": 0, "ghosts": 0, "missed": 0, "regressions": 0}

for img_id in tqdm(coco.getImgIds(), desc="Visual Audit"):
    info = coco.loadImgs(img_id)[0]
    fname = info['file_name']
    path = os.path.join(IMG_ROOT, fname)
    if not os.path.exists(path): continue
    orig = cv2.imread(path)
    if orig is None: continue

    gts = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
    bp, ep = base_map.get(img_id, []), ens_map.get(img_id, [])

    need = {"rescue": False, "missed": False, "regression": False, "ghost": False}
    canv = {k: orig.copy() for k in ["rescue","missed","regression","ghost"]}

    for gt in gts:
        bh = any(get_iou(b['bbox'], gt['bbox']) >= IOU_THRESH for b in bp)
        eh_pred = next((e for e in ep if get_iou(e['bbox'], gt['bbox']) >= IOU_THRESH), None)
        x,y,w,h = [int(v) for v in gt['bbox']]

        if not bh and eh_pred:
            need["rescue"] = True
            cv2.rectangle(canv["rescue"], (x,y), (x+w,y+h), (0,255,0), 2)
            draw_label(canv["rescue"], "GT (Missed by Base)", x, y, (255,255,255), (0,128,0))
            ex,ey,ew,eh_ = [int(v) for v in eh_pred['bbox']]
            cat = CLASS_MAP.get(eh_pred['category_id'], 'Obj')
            cv2.rectangle(canv["rescue"], (ex,ey), (ex+ew,ey+eh_), (255,255,0), 2)
            draw_label(canv["rescue"], f"Ens {cat}: {eh_pred['score']:.2f}", ex, ey+eh_, (0,0,0), (255,255,0))
        elif bh and not eh_pred:
            need["regression"] = True
            cv2.rectangle(canv["regression"], (x,y), (x+w,y+h), (0,255,0), 2)
            draw_label(canv["regression"], "GT (Lost by Ens)", x, y, (255,255,255), (0,128,0))
        elif not bh and not eh_pred:
            need["missed"] = True
            cv2.rectangle(canv["missed"], (x,y), (x+w,y+h), (0,0,255), 2)
            draw_label(canv["missed"], "GT (Missed by BOTH)", x, y, (255,255,255), (0,0,255))

    for e in ep:
        if e['score'] < GHOST_THRESH: continue
        if not any(get_iou(e['bbox'], g['bbox']) > 0.05 for g in gts):
            need["ghost"] = True
            ex,ey,ew,eh_ = [int(v) for v in e['bbox']]
            cat = CLASS_MAP.get(e['category_id'], 'Obj')
            cv2.rectangle(canv["ghost"], (ex,ey), (ex+ew,ey+eh_), (0,165,255), 2)
            draw_label(canv["ghost"], f"GHOST {cat}: {e['score']:.2f}", ex, ey, (255,255,255), (0,140,255))

    if need["rescue"]:
        cv2.imwrite(os.path.join(DIRS["RESCUES"], f"Rescue_{fname}"), canv["rescue"]); counts["rescues"] += 1
    if need["regression"]:
        cv2.imwrite(os.path.join(DIRS["REGRESSIONS"], f"Regression_{fname}"), canv["regression"]); counts["regressions"] += 1
    if need["missed"]:
        cv2.imwrite(os.path.join(DIRS["BOTH_MISSED"], f"Missed_{fname}"), canv["missed"]); counts["missed"] += 1
    if need["ghost"]:
        cv2.imwrite(os.path.join(DIRS["GHOSTS"], f"Ghost_{fname}"), canv["ghost"]); counts["ghosts"] += 1

print(f"\n=== VISUAL AUDIT COMPLETE ===")
for k, v in counts.items(): print(f"  {k}: {v}")

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


Visual Audit: 100%|██████████████████████████████████████████████████████████████████| 603/603 [00:10<00:00, 55.55it/s]


=== VISUAL AUDIT COMPLETE ===
  rescues: 116
  ghosts: 114
  missed: 115
  regressions: 5
